# Fixed-budget reference augmentation

**Draft; results are generated only after validated HPC evaluations.**

Hypothesis: augmenting the single normal reference with controlled darkening and blur may reduce sensitivity to these perturbations. DINOv2 remains frozen; the four banks have identical patch counts. See [protocol](../protocols/reference_augmentation.md).

## Method
For each upstream rotation, extract the original and selected perturbed views. Sample without replacement to match the original patch count. Normalize the bank and compute 1-NN cosine distances. Reuse each test image's features and mask across configurations.

The same-pipeline rotation-only comparator controls the randomized PCA mask. Archived baseline differences must be disclosed.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT = ROOT / 'reports/reference_augmentation'
assert (REPORT/'per_run.csv').is_file(), 'Full results pending: run the validated HPC pipeline first.'
runs = pd.read_csv(REPORT/'per_run.csv')
assert len(runs) == 900
assert not runs.duplicated(['category','seed','variant','condition']).any()
assert (runs.groupby(['category','seed','condition']).bank_patches.nunique()==1).all()

## Paired macro performance and ablation

In [ ]:
macro = pd.read_csv(REPORT/'macro.csv')
for metric in ['image_auroc','pixel_aupro']:
    table = macro.pivot(index='condition', columns='variant', values=metric)*100
    display(table.round(2))
    ax = table.plot.bar(figsize=(10,4), rot=25)
    ax.set_ylabel(metric + ' (%)')
    ax.set_title('Fixed-budget reference augmentation')
    plt.tight_layout(); plt.show()

## Factorial interaction and robustness gap
Positive interaction is a descriptive difference-in-differences, not proof of independent causal effects.

In [ ]:
interaction = pd.read_csv(REPORT/'factorial_interaction_pp.csv')
display(interaction.groupby('condition')[['image_auroc','pixel_aupro']].mean().round(3))
for metric in ['image_auroc','pixel_aupro']:
    display(macro.pivot(index='condition',columns='variant',values=metric+'_robustness_gap_pp').round(3))

## Limitations
The candidate uses the same benchmark as the robustness study; exploratory, not independently confirmatory. Equal bank size controls cardinality, not the exact patch composition. Report negative results and clean performance trade-offs.